Az Ollama fusson (ma csak három modellhívás lesz). Futtasd sorban: **Shift + Enter**. Ha valami elakad: **Restart**, aztán **Run All**. A Copilot kódját olvasd el, mielőtt **Keep**-et nyomsz; ha rossz, **Undo**.

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
import pymupdf  # PDF-olvasó csomag

import kurzus
from kurzus import chat, kerdez, utolso_hasznalat
from kurzus import ellenoriz, ellenoriz_feltetel, ellenoriz_fuggveny

## 1. Miért nem adhatjuk oda az egész jelentést?

A gyűjteményünk az MNB 14 Inflációs jelentése (2023 júniusától 2026 szeptemberéig). A PDF-ek az első futáskor töltődnek le, egyenként kb. 2–3 MB.

In [ ]:
forrasok = pd.read_csv("data/mnb/forrasok.csv", sep=";")
forrasok

A 2024. szeptemberi jelentés (az első futáskor letölti):

In [ ]:
sor = forrasok[forrasok["jelentes"] == "2024-09"].iloc[0]
pdf = pymupdf.open(kurzus.letolt(sor["url"], "data/mnb/" + sor["fajl"]))

teljes_szoveg = "".join(oldal.get_text() for oldal in pdf)
print(f"{len(pdf)} oldal, {len(teljes_szoveg)} karakter")

A modell tokent lát, nem karaktert. Egy oldal tokenszámát a modelltől kérdezzük meg: elküldjük, és csak egy tokennyi választ engedünk (`max_tokens=1`). Ebből lesz a karakter/token arány (pár másodperc).

In [ ]:
def tokenszam(szoveg):
    """Hány token a szöveg a helyi modell szerint. Egyetlen tokennyi választ kérünk, hogy gyors legyen."""
    chat([{"role": "user", "content": szoveg}], gondolkodas="none", max_tokens=1)
    return utolso_hasznalat["prompt_tokens"]


minta = pdf[10].get_text()          # a PDF 11. oldala (a számozás 0-tól indul)
minta_tokenek = tokenszam(minta)
arany = len(minta) / minta_tokenek
print(f"{len(minta)} karakter → {minta_tokenek} token, vagyis kb. {arany:.1f} karakter egy token")

### Feladat 1.1 – Copilottal

> Az `arany` (karakter/token) alapján becsüld meg, hány token a `teljes_szoveg`; ez kerüljön a `becsult_tokenek` változóba. Számold ki azt is, hányszor töltené meg a 4096 tokenes kontextusablakot (`ablakok_szama`, tört szám). Írd ki mindkettőt.

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
ellenoriz("becsult_tokenek", vart=len(teljes_szoveg) / arany, tures=1)
ellenoriz("ablakok_szama", vart=len(teljes_szoveg) / arany / 4096, tures=0.1)

### Mi történik, ha mégis beküldjük?

A helyi modell ablaka **4096 token**: ebbe fér a kérdés, a beküldött szöveg és a válasz együtt. Ami nem fér bele, azt az Ollama csendben levágja a kérés elejéről.

Kísérlet: a jelentés első 12 oldala, a végén a kérdés az inflációs célról. A cél a PDF 5. oldalán áll, tehát benne van abban, amit elküldünk (kb. 10 mp). Figyeld a `FIGYELEM` sort.

In [ ]:
KERDES_CEL = ("\n\nCsak egy számot írj: hány százalék az MNB inflációs célja a fenti szöveg szerint? "
              "Ha a fenti szövegben nem szerepel, írj 0-t.")

elso_12_oldal = "".join(pdf[i].get_text() for i in range(12))
print("A célról szóló mondat benne van az elküldött szövegben:", "3 százalékos középtávú cél" in elso_12_oldal)

valasz_12_oldal = kerdez(elso_12_oldal + KERDES_CEL, gondolkodas="none")
csonkolt_tokenek = utolso_hasznalat["prompt_tokens"]
print("A modell válasza:", valasz_12_oldal)
print("Bemeneti tokenek:", csonkolt_tokenek)

Most csak a PDF 5. oldala:

In [ ]:
valasz_5_oldal = kerdez(pdf[4].get_text() + KERDES_CEL, gondolkodas="none")
print("A modell válasza:", valasz_5_oldal)
print("Bemeneti tokenek:", utolso_hasznalat["prompt_tokens"])

### Feladat 1.2 – olvasás

Olvasd el a két feltételt, mielőtt futtatod: melyik mit mér?

In [ ]:
# 1. A 12 oldal (becslés az arányból) több egy ablaknál, a modell mégis egy ablaknál kevesebbet dolgozott fel.
ellenoriz_feltetel("csonkolt_tokenek", lambda t: t < 4096 < len(elso_12_oldal) / arany,
                   "a 12 oldalas kérés nem fért bele a 4096 tokenbe, a modell csak egy részét látta")
# 2. Az 5. oldal szövegében szerepel a 3 százalékos cél, és a modell is ezt mondta.
ellenoriz_feltetel("valasz_5_oldal", lambda v: "3 százalékos" in pdf[4].get_text() and "3" in v,
                   "csak a jó oldallal a modell a dokumentumból válaszol")

## 2. PDF-ből szöveg

A PDF 11. oldala nyersen:

In [ ]:
print(pdf[10].get_text()[:1600])

Keresd meg benne:

- az **élőfejet** (fejezetcím csupa nagybetűvel, „INFLÁCIÓS JELENTÉS • 2024. SZEPTEMBER”);
- az **oldalszámot** egy külön sorban;
- a **sortöréseket** a mondatok közepén;
- a sor végi **elválasztást** (kötőjel a sor végén).

### Feladat 2.1 – olvasás

Milyen nyomtatott oldalszám áll a PDF 11. oldalán? Olvasd ki a fenti kiírásból.

In [ ]:
tipp_2_1 = None  # a None helyére írd a kiolvasott számot

In [ ]:
# Az oldal első néhány sorából az, amelyik csak egy szám.
vart_2_1 = next(int(s) for s in pdf[10].get_text().splitlines()[:5] if s.strip().isdigit())
ellenoriz("tipp_2_1", vart=vart_2_1)

A PDF-oldal és a nyomtatott oldal eltér. Ez a függvény olvassa ki a nyomtatottat:

In [ ]:
def nyomtatott_oldalszam(oldal):
    """Az élőfejben álló nyomtatott oldalszám; ha az oldalon nincs élőfej (borító, belső borító), None."""
    elso_sorok = [s.strip() for s in oldal.get_text().splitlines()[:5]]
    if not any("INFLÁCIÓS JELENTÉS •" in s for s in elso_sorok):
        return None                  # a borítón egy évszám is állhat külön sorban: az nem oldalszám
    for sor in elso_sorok:
        if sor.isdigit():
            return int(sor)
    return None


# Mindegyik jelentésben ugyanannyi-e az eltérés?
for s in forrasok.itertuples():
    jelentes_pdf = pymupdf.open(kurzus.letolt(s.url, "data/mnb/" + s.fajl))
    elteres = {i + 1 - nyomtatott_oldalszam(o) for i, o in enumerate(jelentes_pdf) if nyomtatott_oldalszam(o)}
    print(s.jelentes, "PDF-oldal − nyomtatott oldal:", elteres)

### Takarítás

A `tisztit` (1) összevonja a sor végi elválasztást, ha kisbetű jön utána, (2) törli a tartalomjegyzék pontsorait, (3) a sortöréseket és a többszörös szóközöket egy szóközre cseréli.

In [ ]:
def tisztit(szoveg):
    """PDF-ből kinyert szöveg takarítása: elválasztás, pontsorok, sortörések, szóközök."""
    szoveg = re.sub(r"-\n(?=[a-záéíóöőúüű])", "", szoveg)  # sor végi elválasztás: ren-|delkezésére
    szoveg = re.sub(r"\.{4,}", " ", szoveg)                # tartalomjegyzék pontsora
    szoveg = szoveg.replace("\n", " ")
    return re.sub(r"\s+", " ", szoveg).strip()


print(tisztit("a ren-\ndelkezésére álló\n  eszközökkel"))

### Feladat 2.2 – olvasás

Mit ad vissza a `tisztit` erre a két sorra: *„Az erőteljes reálbér-”* / *„növekedés és stabil foglalkoztatás”*? Írd be idézőjelek közé, pontosan.

In [ ]:
tipp_2_2 = None  # a None helyére írd a tipped, idézőjelek közé

In [ ]:
ellenoriz("tipp_2_2", vart=tisztit("Az erőteljes reálbér-\nnövekedés és stabil foglalkoztatás"))

Mit rontott el a szabály, és baj-e ez a keresésnél?

## 3. Bekezdések és oldalszámok

A `get_text("blocks")` nem sorokat, hanem blokkokat ad: nagyjából bekezdéseket. Az `oldal_bekezdesei` kidobja, ami nem folyó szöveg: az élőfejet és az élőlábat, a csupa nagybetűs fejezetcímet, és a grafikonok feliratait (öt szónál rövidebb, vagy nincs benne három betű egymás után). A `jelentes_oldalai` minden oldalhoz eltárolja a jelentést, a PDF-oldalt és a nyomtatott oldalt.

In [ ]:
FEJLEC_JELEK = ("INFLÁCIÓS JELENTÉS •", "MAGYAR NEMZETI BANK")


def oldal_bekezdesei(oldal):
    """Az oldal bekezdései takarítva, élőfej, grafikonfeliratok és oldalszám nélkül."""
    bekezdesek = []
    for blokk in oldal.get_text("blocks"):
        szoveg = tisztit(blokk[4])               # a blokk ötödik eleme a szöveg
        if any(jel in szoveg for jel in FEJLEC_JELEK):
            continue                             # élőfej, élőláb
        if szoveg.isupper():
            continue                             # csupa nagybetűs fejezetcím
        if len(szoveg.split()) < 5 or not re.search(r"[a-zA-ZáéíóöőúüűÁÉÍÓÖŐÚÜŰ]{3}", szoveg):
            continue                             # grafikon tengelye, jelmagyarázat
        bekezdesek.append(szoveg)
    return bekezdesek


def jelentes_oldalai(pdf_utvonal, jelentes):
    """Egy jelentés oldalai: jelentés, PDF-oldalszám, nyomtatott oldalszám és a bekezdések."""
    oldalak = []
    for pdf_oldal, oldal in enumerate(pymupdf.open(pdf_utvonal), start=1):
        oldalak.append({"jelentes": jelentes, "pdf_oldal": pdf_oldal,
                        "oldal": nyomtatott_oldalszam(oldal), "bekezdesek": oldal_bekezdesei(oldal)})
    return oldalak


oldalak = jelentes_oldalai("data/mnb/" + sor["fajl"], "2024-09")
pelda = oldalak[10]
print("PDF", pelda["pdf_oldal"], "→ nyomtatott", pelda["oldal"], "oldal,", len(pelda["bekezdesek"]), "bekezdés")
for b in pelda["bekezdesek"][:3]:
    print("-", b[:200], "…")

### Feladat 3.1 – Copilottal

> Az `oldalak` listában keresd meg az első oldalt, amelynek valamelyik bekezdésében szerepel a „középtávú cél” kifejezés. A PDF-oldalszámát tedd a `cel_pdf_oldal`, a nyomtatott oldalszámát a `cel_oldal` változóba, és írd ki mindkettőt.

Olvasd el: megáll-e a ciklus az első találatnál?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
# A várt oldal a nyers PDF-ből, a bekezdésektől függetlenül.
vart_cel_oldal = next(i + 1 for i, o in enumerate(pdf) if "középtávú cél" in tisztit(o.get_text()))
ellenoriz("cel_pdf_oldal", vart=vart_cel_oldal)

## 4. Darabolás: három stratégia

A 2024. szeptemberi jelentésen, oldalanként (így minden darabnak egy oldalszáma van):

1. **fix méret:** 1000 karakterenként vágunk;
2. **fix méret átfedéssel:** minden darab 200 karakterrel az előző vége előtt kezdődik;
3. **bekezdésenként:** egész bekezdéseket fűzünk össze, legfeljebb 1000 karakterig; a túl hosszú bekezdést mondathatáron vágjuk.

1000 karakter kb. 350–400 token: öt ilyen darab a kérdéssel együtt elfér 3000 tokenben.

In [ ]:
def darabol_fix(szoveg, meret=1000, atfedes=0):
    """Fix méretű darabok (karakterben); az egymást követő darabok `atfedes` karakternyit átfednek."""
    lepes = meret - atfedes
    return [szoveg[i:i + meret] for i in range(0, max(len(szoveg) - atfedes, 1), lepes)]

### Feladat 4.1 – olvasás

Hány darabot ad vissza: `darabol_fix("abcdefghij", meret=4, atfedes=2)`?

In [ ]:
tipp_4_1 = None  # a None helyére írd a tipped

In [ ]:
print(darabol_fix("abcdefghij", meret=4, atfedes=2))
ellenoriz("tipp_4_1", vart=len(darabol_fix("abcdefghij", meret=4, atfedes=2)))

### Mondathatár

A túl hosszú bekezdést mondatonként vágjuk. Mondatvég: pont (kérdőjel, felkiáltójel), szóköz, nagybetű vagy számjegy. **Kivétel**, ha a pont előtt szám („2025. I. negyedév”), római szám vagy egy betű („III. Frigyes”, „Kr. e.”) vagy rövidítés („pl.”, „stb.”) áll. Futtasd, és nézd meg, hol vágott, hol nem:

In [ ]:
ROVIDITESEK = {"pl", "kb", "stb", "ill", "ún", "vö", "szp", "Kr", "dr"}   # szp. = százalékpont


def mondatokra(szoveg):
    """Mondatokra bontja a szöveget. Mondatvég: pont, kérdőjel vagy felkiáltójel, utána szóköz és
    nagybetű vagy számjegy; kivéve, ha a pont előtt szám, római szám, egyetlen betű vagy rövidítés áll."""
    mondatok = []
    for resz in re.split(r"(?<=[.!?]) (?=[A-ZÁÉÍÓÖŐÚÜŰ0-9])", szoveg):
        elozo_szo = mondatok[-1].split()[-1].rstrip(".") if mondatok else ""
        if elozo_szo.isdigit() or re.fullmatch(r"[IVX]+|\w", elozo_szo) or elozo_szo in ROVIDITESEK:
            mondatok[-1] += " " + resz          # nem mondatvég: visszaragasztjuk az előzőhöz
        else:
            mondatok.append(resz)
    return mondatok


# Kitalált példamondatok (a számok csak a példa kedvéért):
pelda = ("Az infláció kb. 3,7% volt, és a 3. negyedévben pl. Ausztriában is csökkent. "
         "A 2025. I. negyedévben drágult az élelmiszer, az energia stb. "
         "2026-ban lassulhat az áremelkedés. A hátsó borítón I. Mátyás király áll.")
for mondat in mondatokra(pelda):
    print("-", mondat)

Hol nem vágott, és miért? Hol kellett volna mégis vágnia? A bekezdésenkénti darabolás ezzel:

In [ ]:
def darabol_bekezdesenkent(bekezdesek, max_hossz=1000):
    """Egész bekezdéseket fűz össze, amíg a darab el nem érné a `max_hossz` karaktert. A túl hosszú
    bekezdést mondatokra bontja; ha egyetlen mondat is túl hosszú, azt szavakra."""
    reszek = []                                    # a legnagyobb egységek, amelyek egyenként beférnek
    for bekezdes in bekezdesek:
        if len(bekezdes) <= max_hossz:
            reszek.append(bekezdes)
            continue
        for mondat in mondatokra(bekezdes):        # túl hosszú bekezdés: mondathatáron vágjuk
            if len(mondat) <= max_hossz:
                reszek.append(mondat)
            else:
                reszek.extend(mondat.split())      # végső eset: szóhatáron, de szót nem vágunk ketté

    darabok = []
    aktualis = ""
    for resz in reszek:
        if aktualis and len(aktualis) + 1 + len(resz) > max_hossz:
            darabok.append(aktualis)               # nem fér bele: új darab kezdődik
            aktualis = resz
        else:
            aktualis = (aktualis + " " + resz).strip()
    if aktualis:
        darabok.append(aktualis)
    return darabok

A három stratégia a teljes jelentésen. A „mondat közepén kezdődik” azt számolja, hány darab kezdődik kisbetűvel.

In [ ]:
strategiak = {
    "fix 1000": [d for o in oldalak for d in darabol_fix(" ".join(o["bekezdesek"]), 1000) if d.strip()],
    "fix 1000, átfedés 200": [d for o in oldalak for d in darabol_fix(" ".join(o["bekezdesek"]), 1000, 200) if d.strip()],
    "bekezdésenként, max 1000": [d for o in oldalak for d in darabol_bekezdesenkent(o["bekezdesek"], 1000)],
}

sorok = []
for nev, darabok in strategiak.items():
    hosszak = [len(d) for d in darabok]
    sorok.append({"stratégia": nev, "darab": len(darabok),
                  "átlagos hossz": round(sum(hosszak) / len(hosszak)), "leghosszabb": max(hosszak),
                  "mondat közepén kezdődik": sum(d[0].islower() for d in darabok)})
osszehasonlitas = pd.DataFrame(sorok).set_index("stratégia")
osszehasonlitas

Egy vágás közelről (PDF 11. oldal):

In [ ]:
szoveg_11 = " ".join(oldalak[10]["bekezdesek"])

fix = darabol_fix(szoveg_11, 1000)
print("FIX, az 1. darab vége:     …" + fix[0][-120:])
print("FIX, a 2. darab eleje:      " + fix[1][:120] + "…\n")

atfedo = darabol_fix(szoveg_11, 1000, 200)
print("ÁTFEDÉS, a 2. darab eleje:  " + atfedo[1][:120] + "…\n")

bekezdes = darabol_bekezdesenkent(oldalak[10]["bekezdesek"], 1000)
print("BEKEZDÉS, az 1. darab vége: …" + bekezdes[0][-120:])
print("BEKEZDÉS, a 2. darab eleje: " + bekezdes[1][:120] + "…")

Hol vágott a fix méret? Mi van az átfedéses második darab elején? Hol ér véget a bekezdéses darab? Honnan jön a bekezdésesnél a néhány kisbetűs kezdet? A túl kicsi darabban elvész az összefüggés, a túl nagy darabnál pontatlan lesz a keresés.

### Feladat 4.2 – Copilottal

> Az `arany` (karakter/token) és a `strategiak["bekezdésenként, max 1000"]` lista alapján számold ki, hány token egy darab átlagosan. Az eredmény az `atlagos_token` változóba kerüljön.

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
_bd = strategiak["bekezdésenként, max 1000"]
ellenoriz("atlagos_token", vart=sum(len(d) for d in _bd) / len(_bd) / arany, tures=1)

### Feladat 4.3 – hibakeresés

Ez a függvény lefut, de két darabot nem lehetne megkülönböztetni. Futtasd, és nézd meg, mit ír ki.

In [ ]:
def darabok_keszitese_v1(oldalak, max_hossz=1000):
    darabok = []
    for o in oldalak:
        for szoveg in darabol_bekezdesenkent(o["bekezdesek"], max_hossz):
            darabok.append({"id": f'{o["jelentes"]}_p{o["pdf_oldal"]:03d}', "jelentes": o["jelentes"],
                            "pdf_oldal": o["pdf_oldal"], "oldal": o["oldal"], "szoveg": szoveg})
    return darabok


proba = darabok_keszitese_v1(oldalak)
print(len(proba), "darab,", len({d["id"] for d in proba}), "különböző azonosító")
print(proba[20])

Írd meg a `darabok_keszitese(oldalak, max_hossz=1000)` függvényt egyedi azonosítókkal: a végére kerüljön a darab sorszáma az oldalon belül (`2024-09_p011_0`, `2024-09_p011_1`, …; tipp: `enumerate`).

In [ ]:
# Ide kerül a javítás

In [ ]:
if "darabok_keszitese" in globals():
    _d = darabok_keszitese(oldalak)
    egyedi_es_teljes = len({x["id"] for x in _d}) == len(_d) == len(proba) and _d[0]["id"].endswith("_0")
else:
    egyedi_es_teljes = None
ellenoriz_feltetel("egyedi_es_teljes", lambda x: x, "minden darab megvan, és mindegyiknek egyedi az azonosítója")

## 5. Mind a 14 jelentés darabolása és mentése

A darabok a `data/cache/darabok.json`-be kerülnek; a következő alkalommal innen dolgozunk. A cellában a `darabok_keszitese` javított változata is benne van. Kb. 5–10 mp.

In [ ]:
def darabok_keszitese(oldalak, max_hossz=1000):
    """Egy jelentés darabjai; minden darab megőrzi a jelentést és mindkét oldalszámot."""
    darabok = []
    for o in oldalak:
        for j, szoveg in enumerate(darabol_bekezdesenkent(o["bekezdesek"], max_hossz)):
            darabok.append({"id": f'{o["jelentes"]}_p{o["pdf_oldal"]:03d}_{j}', "jelentes": o["jelentes"],
                            "pdf_oldal": o["pdf_oldal"], "oldal": o["oldal"], "szoveg": szoveg})
    return darabok


DARABOK_FAJL = Path("data/cache/darabok.json")

darabok = []
for s in forrasok.itertuples():
    pdf_utvonal = kurzus.letolt(s.url, "data/mnb/" + s.fajl)
    darabok += darabok_keszitese(jelentes_oldalai(pdf_utvonal, s.jelentes))

DARABOK_FAJL.parent.mkdir(parents=True, exist_ok=True)
DARABOK_FAJL.write_text(json.dumps(darabok, ensure_ascii=False), encoding="utf-8")
print(f"{len(darabok)} darab, {sum(len(d['szoveg']) for d in darabok)} karakter → {DARABOK_FAJL}")
pd.Series([d["jelentes"] for d in darabok]).value_counts().sort_index()

In [ ]:
ellenoriz_feltetel("darabok", lambda ds: len({d["id"] for d in ds}) == len(ds)
                   and all(len(d["szoveg"]) <= 1000 and d["jelentes"] and d["pdf_oldal"] for d in ds)
                   and json.loads(DARABOK_FAJL.read_text(encoding="utf-8")) == ds,
                   "egyedi azonosítók, legfeljebb 1000 karakter, forrás mindenhol, a fájl a listát tartalmazza")

Egy darab (a PDF 11. oldal első darabja):

In [ ]:
next(d for d in darabok if d["id"] == "2024-09_p011_0")

### Feladat 5.1 – Copilottal

> A `darabok` listában keresd meg azt a darabot, amelynek szövegében szerepel: „2024-ben az infláció 3,7 százalékon alakult”. Tedd a `talalat` változóba, és írd ki a jelentést, a nyomtatott oldalszámot és a PDF-oldalszámot.

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
# A várt forrás közvetlenül a PDF-ekből (a darabolástól függetlenül): melyik jelentés melyik oldala.
MONDAT = "2024-ben az infláció 3,7 százalékon alakult"
vart_forras = None
for s in forrasok.itertuples():
    for o in pymupdf.open("data/mnb/" + s.fajl):
        if MONDAT in tisztit(o.get_text()):
            vart_forras = (s.jelentes, nyomtatott_oldalszam(o))
print("A PDF-ekben:", vart_forras)
ellenoriz_feltetel("talalat", lambda t: (t["jelentes"], t["oldal"]) == vart_forras,
                   "a darab forrása egyezik a PDF-ben talált jelentéssel és oldallal")

## 6. A projektben

A projektcsomagodon ugyanezt kell majd megcsinálni (élőfej, oldalszám, takarítás, darabolás). Részletek: a diákon és a jegyzetben.